<a href="https://colab.research.google.com/github/amairagoel/Audio-to-Synced-Translated-Lyrics/blob/main/main.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from fastapi import FastAPI, Depends, HTTPException, Security, status, BackgroundTasks, Query
from fastapi.security.api_key import APIKeyHeader
from pydantic import BaseModel, Field, HttpUrl
from typing import List, Optional
import uvicorn

from database import init_db, get_db, B2BLyricsCache
import service

app = FastAPI(
    title="Lyrical-AI Enterprise B2B API",
    description="High-throughput asynchronous slang-aware lyric translation API layer.",
    version="2.0.0"
)

# Enforce secure header keys in standard API parameters
API_KEY_NAME = "X-Enterprise-Token"
api_key_header = APIKeyHeader(name=API_KEY_NAME, auto_error=True)

# Seed live clients dataset map for runtime evaluation
VALID_ENTERPRISE_CLIENTS = {
    "b2b_live_sony_latin_88290": "Sony Music Latin",
    "b2b_live_distrokid_7716a": "DistroKid Engineering Team",
    "b2b_live_musixmatch_1102f": "Musixmatch Localization Lab"
}

def authenticate_client(token: str = Depends(api_key_header)) -> str:
    if token not in VALID_ENTERPRISE_CLIENTS:
        raise HTTPException(
            status_code=status.HTTP_401_UNAUTHORIZED,
            detail="Invalid, missing, or revoked corporate API token authorization token."
        )
    return VALID_ENTERPRISE_CLIENTS[token]

# --- Pydantic Structural Validations ---
class LyricLine(BaseModel):
    timestamp: str = Field(..., example="[01:04.50]")
    text: str = Field(..., example="Tranquila, baby, que la vida es un ciclo")

class BatchTranslationTask(BaseModel):
    track_id: str = Field(..., description="Unique global identification code (ISRC, Spotify URI, or internal DB key).")
    artist: str = Field(..., example="Bad Bunny")
    title: str = Field(..., example="Monaco")
    callback_url: str = Field(..., description="The endpoint on your systems we push completed data back to.")
    raw_lyrics_override: Optional[List[LyricLine]] = Field(
        None,
        description="Optional: Provide raw source text directly. If empty, the API auto-fetches from public registries."
    )

class TaskAcceptedResponse(BaseModel):
    status: str = "Accepted"
    message: str
    track_id: str
    client: str

# --- Asynchronous Worker Logic Loop ---
async def process_translation_async(task: BatchTranslationTask, client_name: str):
    """Handles the processing sequence safely away from the main user request line."""
    db_session = next(get_db())
    try:
        artist_norm = task.artist.strip().lower()
        title_norm = task.title.strip().lower()
        final_translated_lrc = ""

        # Step 1: Evaluate Database Cache Hits
        cached_record = db_session.query(B2BLyricsCache).filter(
            B2BLyricsCache.artist == artist_norm,
            B2BLyricsCache.title == title_norm
        ).first()

        if cached_record:
            final_translated_lrc = cached_record.translated_lrc
            print(f"[Cache Hit] Retrieved processing logs for Track: {task.track_id}")
        else:
            # Step 2: Extract text input blocks
            source_lrc_string = ""
            if task.raw_lyrics_override:
                source_lrc_string = "\n".join([f"{line.timestamp} {line.text}" for line in task.raw_lyrics_override])
            else:
                source_lrc_string = await service.fetch_lrclib_synced(task.artist, task.title)

            if not source_lrc_string.strip():
                # Notify webhook endpoint that translation failed due to missing source lyrics
                error_payload = {"track_id": task.track_id, "status": "failed", "reason": "No source lyrics found."}
                await service.dispatch_webhook_callback(task.callback_url, error_payload)
                return

            # Step 3: Run contextual slang translator
            final_translated_lrc = service.translate_lrc_contextually(source_lrc_string)

            # Step 4: Write to Database Cache for future hits
            new_cache_entry = B2BLyricsCache(
                artist=artist_norm,
                title=title_norm,
                original_lrc=source_lrc_string,
                translated_lrc=final_translated_lrc
            )
            db_session.add(new_cache_entry)
            db_session.commit()

        # Step 5: Convert the raw translated LRC text blocks back into a clean JSON array
        formatted_lines = []
        for line in final_translated_lrc.split("\n"):
            if line.strip() and line.startswith("["):
                parts = line.split("] ", 1)
                if len(parts) == 2:
                    formatted_lines.append({"timestamp": parts[0] + "]", "text": parts[1].strip()})

        # Step 6: Dispatch outbound payload to corporate endpoint
        webhook_payload = {
            "track_id": task.track_id,
            "status": "success",
            "processed_by": "Lyrical-AI Core Eng v2",
            "client_account": client_name,
            "artist": task.artist,
            "title": task.title,
            "translated_lyrics": formatted_lines
        }
        await service.dispatch_webhook_callback(task.callback_url, webhook_payload)

    except Exception as worker_err:
        print(f"[Worker Error] Internal Pipeline Crash for track {task.track_id}: {str(worker_err)}")
    finally:
        db_session.close()

# --- Public API Endpoint Routing Interfaces ---
@app.post(
    "/api/v2/enterprise/translate",
    status_code=status.HTTP_202_ACCEPTED,
    response_model=TaskAcceptedResponse,
    summary="Ingest music tracks for high-performance localized slang translation via webhooks."
)
async def queue_catalog_translation(
    task: BatchTranslationTask,
    background_tasks: BackgroundTasks,
    client_org: str = Depends(authenticate_client)
):
    """
    Enterprise ingestion interface. Securely receives target parameters,
    returns an immediate 202 receipt code, and executes processing tasks
    asynchronously inside background workers.
    """
    # Throw processing workload down to background thread execution engines
    background_tasks.add_task(process_translation_async, task, client_org)

    return TaskAcceptedResponse(
        message="Transaction block accepted into background queue. Results will be emitted via your callback URL.",
        track_id=task.track_id,
        client=client_org
    )

@app.on_event("startup")
def startup_db_hook():
    init_db()

if __name__ == "__main__":
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=True)